# Avon River analysis — 01 Data understanding and validation

Eric Gomez · October 2026

An environmental organisation monitors three sites on the Avon River (AV-1, AV-2 and AV-3). These notebooks
analyse its records from October to December 2023: water temperature, pH and dissolved oxygen (DO) at each
visit, and the count and mean size of each fish species surveyed.

This notebook profiles the raw file, checks it record by record, and decides how each problem is handled.
Every decision names the Excel rows it affects and the evidence behind it. Where the evidence leaves room
for doubt, the alternative is carried into the sensitivity analysis in `03`.

| Notebook | Contents |
|---|---|
| `01` | Data understanding and validation (this notebook) |
| `02` | Cleaning, derived measures, the data validation and management matrix, and figures |
| `03` | Hypothesis tests, sensitivity analysis, and whether a predictive model is justified |

The raw spreadsheet is not included in the repository; the executed notebooks show every output.

In [1]:
import sys, pathlib

ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.visualization.theme import PALETTE, save_fig, set_plot_style
set_plot_style()

# Raw monitoring file (not included in this repository).
DATASET = 'Data_Set_Assignmnet_1 - 0426.xlsx'

## The question

What do three sites' water-quality and fish records from October to December 2023 show? In particular:

- which environmental pressures on *native* fish the data support;
- whether any water-quality ↔ fish relationship can be detected;
- what further data would settle the questions these records cannot.

Four things the file does not record could change the answers:

- **Sampling times.** DO and temperature swing through the day.
- **Site order** along the river.
- **Survey effort.** Without it, one survey's count can't be compared with another's.
- **The origin of the copied rows.** Re-entries, or real repeat samples? (Checked below.)

In [2]:
from src.data.loading import list_raw, load_raw

print('files in data/raw/:', list_raw())
raw = load_raw(DATASET, header=None)            # read without a header first, to see the layout
print('sheet shape:', raw.shape)
raw.head(4)

files in data/raw/: ['.DS_Store', 'Data_Set_Assignmnet_1 - 0426.xlsx']
sheet shape: (102, 11)


,0,1,2,3,4,5,6,7,8,9,10
0,Water Quality Data,NaN,NaN,NaN,NaN,NaN,Fish Population Data,NaN,NaN,NaN,NaN
1,Site ID,Date,Temperature (°C),pH,Dissolved Oxygen (mg/L),NaN,Site ID,Date,Species,Count,Avg. Size (cm)
2,AV-1,2023-11-06 00:00:00,13.5,7.88,7.9,NaN,AV-1,2023-11-06 00:00:00,Brown Trout,12,18.3
3,AV-2,2023-11-02 00:00:00,16.8,7.33,8.2,NaN,AV-2,2023-11-02 00:00:00,Shortfin Eel,7,42.1


### Interpretation

- **Layout.** One sheet holds **two tables side by side**:
  - "Water Quality Data" in columns A–E;
  - "Fish Population Data" in columns G–K;
  - column F is empty between them.
- **Header and data rows.** Row 1 holds the table titles, row 2 the column headers, and data starts in row 3.
- **Approach.** The sheet is split into two tables, and each record keeps its **Excel row number**, so every
  decision below can name the exact row it affects.

In [3]:
def side_table(cols):
    """One of the two side-by-side tables, with its own header and each record's Excel row."""
    t = raw.iloc[2:, cols].copy()
    t.columns = raw.iloc[1, cols].tolist()
    t.insert(0, 'excel_row', t.index + 1)       # pandas row 0 is Excel row 1
    return t.dropna(how='all', subset=t.columns[1:]).reset_index(drop=True)


wq = side_table(slice(0, 5))      # Water Quality Data, columns A-E
fish = side_table(slice(6, 11))   # Fish Population Data, columns G-K

print('water quality:', wq.shape, ' Excel rows', wq.excel_row.min(), '-', wq.excel_row.max())
print('fish:         ', fish.shape, ' Excel rows', fish.excel_row.min(), '-', fish.excel_row.max())
paired = [(a == b) and (c == d) for a, b, c, d in zip(wq['Site ID'], fish['Site ID'], wq['Date'], fish['Date'])]
first_unpaired = fish.excel_row[paired.index(False)] if False in paired else None
print('rows where both tables show the same site and date, side by side:', sum(paired),
      '| first row where they differ:', first_unpaired)
display(wq.head(3)); display(fish.head(3))

water quality: (100, 6)  Excel rows 3 - 102
fish:          (90, 6)  Excel rows 3 - 92
rows where both tables show the same site and date, side by side: 71 | first row where they differ: 74


,excel_row,Site ID,Date,Temperature (°C),pH,Dissolved Oxygen (mg/L)
0,3,AV-1,2023-11-06 00:00:00,13.5,7.88,7.9
1,4,AV-2,2023-11-02 00:00:00,16.8,7.33,8.2
2,5,AV-3,2023-10-21 00:00:00,15.1,7.6,7.5


,excel_row,Site ID,Date,Species,Count,Avg. Size (cm)
0,3,AV-1,2023-11-06 00:00:00,Brown Trout,12,18.3
1,4,AV-2,2023-11-02 00:00:00,Shortfin Eel,7,42.1
2,5,AV-3,2023-10-21 00:00:00,Inanga,25,7.4


### Interpretation

- **Size.** 100 water-quality records (Excel rows 3–102) and 90 fish records (rows 3–92).
- **How the tables pair up.**
  - In rows 3–73, each water-quality reading sits beside the fish record from the same site and date.
    The sheet was built as one row per site visit.
  - From row 74 the two tables drift apart. Below row 73 the sheet holds copies of the rows above,
    some exact and some with a date or a value changed (see *Data-integrity errors* below).
- **What one record is.** One water-quality record is one reading at one site on one date. One fish
  record is one species' count and mean size on one visit.

In [4]:
from src.data.quality import profile, schema_report

for name, frame in [('water quality', wq), ('fish', fish)]:
    print(f'\n--- {name} ---')
    profile(frame)
    display(schema_report(frame))


--- water quality ---
rows: 100   columns: 6
memory: 21.0 KB

dtypes:
object    4
int64     1
str       1


,column,dtype,non_null,missing,missing_pct,unique,example
0,excel_row,int64,100,0,0.0,100,3
1,Site ID,str,100,0,0.0,3,AV-1
2,Date,object,100,0,0.0,54,2023-11-06 00:00:00
3,Temperature (°C),object,100,0,0.0,55,13.5
4,pH,object,99,1,1.0,50,7.88
5,Dissolved Oxygen (mg/L),object,99,1,1.0,28,7.9



--- fish ---
rows: 90   columns: 6
memory: 21.5 KB

dtypes:
object    3
str       2
int64     1


,column,dtype,non_null,missing,missing_pct,unique,example
0,excel_row,int64,90,0,0.00,90,3
1,Site ID,str,90,0,0.00,3,AV-1
2,Date,object,90,0,0.00,54,2023-11-06 00:00:00
3,Species,str,88,2,2.22,4,Brown Trout
4,Count,object,90,0,0.00,29,12
5,Avg. Size (cm),object,89,1,1.11,59,18.3


### Interpretation

- **Types.** The date and measurement columns load as generic `object`, because the header rows were read
  as data. Site ID loads as text. `02` sets each column's type explicitly.
- **Units.** Temperature is in °C, DO in mg/L, pH has no unit, and size is in cm. Count is the number of
  fish per survey; the survey effort behind it isn't recorded.

## Validation approach

Each problem found below gets one of four classifications. Each classification has a test and a default
action.

| Classification | Test | Default action |
|---|---|---|
| **Critical anomaly** | The value is physically impossible | Remove the record; it cannot enter any analysis |
| **Data-integrity error** | The record is provably not an independent observation: a copy of a record above (verbatim or altered), or a date that does not exist | Remove it; where it copies a record, keep the source |
| **Statistical outlier** | Extreme by IQR or z-score, physically possible, and not provably an error | Keep, unless the record is suspect on other grounds; either way, test the alternative in the sensitivity analysis |
| **Missing data** | A blank cell | Recover the value from a copy if one exists; infer it only if a single value fits, and flag it; otherwise leave it blank (no imputation) |

**Which class applies.** A record takes the first class in the table that fits.
- 200.7 °C and 79.9 mg/L are critical anomalies, although both sit in altered copies.
- The 3000 count and the 4.9 cm eel are possible values in proven copies, so they are data-integrity errors.
- Row 94 (DO 12.9 mg/L) is a statistical outlier, because nothing proves it is a copy. That is also why its
  removal is tested both ways in the sensitivity analysis.

Two readings for the same site and day are not an error in themselves; each pair is resolved by a stated rule.

The checks below run from format (dates and numbers that fail to parse) through missing values and copies
to the values themselves. `02` summarises every decision, with its impact, in the data validation and
management matrix.

In [5]:
# What fails to parse? Reported here; nothing is fixed in this notebook.
def unparseable_dates(t):
    d = pd.to_datetime(t['Date'], errors='coerce')
    return t.loc[d.isna() & t['Date'].notna(), ['excel_row', 'Site ID', 'Date']]

def unparseable_numbers(t, cols):
    out = []
    for c in cols:
        n = pd.to_numeric(t[c], errors='coerce')
        out += [(int(r), c, v) for r, v in zip(t.loc[n.isna() & t[c].notna(), 'excel_row'], t.loc[n.isna() & t[c].notna(), c])]
    return out

print('water-quality dates that do not parse:'); display(unparseable_dates(wq))
print('fish dates that do not parse:'); display(unparseable_dates(fish))
print('non-numeric values:', unparseable_numbers(wq, ['Temperature (°C)', 'pH', 'Dissolved Oxygen (mg/L)'])
      + unparseable_numbers(fish, ['Count', 'Avg. Size (cm)']))

water-quality dates that do not parse:


,excel_row,Site ID,Date


fish dates that do not parse:


,excel_row,Site ID,Date
71,74,AV-3,11/31/2023


non-numeric values: []


### Interpretation

- **One invalid date.** Fish row 74 holds the text `11/31/2023`. There is no 31 November, so the date
  can't be parsed.
- **Everything else parses.** All other dates and every number are valid, so the remaining problems are
  in the values themselves, not their format.

In [6]:
from src.data.quality import missing_report

for name, frame in [('water quality', wq), ('fish', fish)]:
    print(f'--- {name} ---')
    display(missing_report(frame))
    display(frame[frame.isna().any(axis=1)])

--- water quality ---
complete rows: 98 of 100 (98.0%)


,missing,missing_pct
pH,1,1.0
Dissolved Oxygen (mg/L),1,1.0


,excel_row,Site ID,Date,Temperature (°C),pH,Dissolved Oxygen (mg/L)
57,60,AV-3,2023-12-20 00:00:00,18.1,NaN,7.1
95,98,AV-2,2023-10-31 00:00:00,14.6,7.41,NaN


--- fish ---
complete rows: 87 of 90 (96.7%)


,missing,missing_pct
Species,2,2.22
Avg. Size (cm),1,1.11


,excel_row,Site ID,Date,Species,Count,Avg. Size (cm)
42,45,AV-2,2023-10-24 00:00:00,NaN,26,8.1
65,68,AV-3,2023-12-04 00:00:00,Longfin Eel,7,NaN
77,80,AV-2,2023-10-24 00:00:00,NaN,26,8.1


## Missing data

| Excel row(s) | Missing | Decision | Evidence |
|---|---|---|---|
| WQ 98 (AV-2, 31 Oct) | DO | **not a gap: the record is a copy** (see *Data-integrity errors*) | Row 98 repeats row 30 (same visit, temperature and pH) with the DO left blank; row 30 has DO = 8.2 |
| WQ 60 (AV-3, 20 Dec) | pH | **leave blank** | One value. pH is a secondary variable, and an imputed value would add nothing a test could use |
| Fish 68 (AV-3, 4 Dec, longfin eel ×7) | size | **leave blank** | Size is not used in any test |
| Fish 45, copied at 80 (AV-2, 24 Oct, ×26, 8.1 cm) | species | **infer īnanga, flagged `species_inferred`** | 8.1 cm sits inside the īnanga size range and outside every other species' range (shown in `02`) |

- **Mechanism.** Nothing suggests the gaps depend on the missing values themselves, so missing-not-at-random
  is unlikely. With one cell each, no choice here moves the analysis.
- **Alternative tested.** Dropping the fish 45 record instead of inferring its species removes one of 25
  īnanga records. The sensitivity analysis in `03` runs both versions.

In [7]:
from src.data.quality import duplicate_report

DATA_WQ = ['Site ID', 'Date', 'Temperature (°C)', 'pH', 'Dissolved Oxygen (mg/L)']
DATA_FISH = ['Site ID', 'Date', 'Species', 'Count', 'Avg. Size (cm)']
KEY_WQ = ['Site ID', 'Date']                      # one reading per site per day, if nothing is repeated
KEY_FISH = ['Site ID', 'Date', 'Species']         # one record per species per visit

def copies(t, cols):
    """Groups of rows identical in every data column: [first row, its repeats]."""
    g = t.groupby(cols, dropna=False, sort=False)['excel_row'].apply(list)
    return g[g.map(len) > 1].sort_values().tolist()

def conflicts(t, key, cols):
    """Different records sharing one key (after collapsing exact copies)."""
    d = t.drop_duplicates(cols)
    return d[d.duplicated(key, keep=False)].sort_values(['Site ID', 'excel_row'])

for name, t, key, cols in [('water quality', wq, KEY_WQ, DATA_WQ), ('fish', fish, KEY_FISH, DATA_FISH)]:
    print(f'--- {name} ---')
    duplicate_report(t[cols], subset=key)
    groups = copies(t, cols)
    repeats = sorted(r for g in groups for r in g[1:])
    print(f'exact copies: {len(repeats)} rows repeat an earlier row -> {repeats}')
    print('conflicting records on the same key:')
    display(conflicts(t, key, cols))

--- water quality ---
exact duplicate rows: 25
duplicate on ['Site ID', 'Date']: 29
exact copies: 25 rows repeat an earlier row -> [47, 74, 75, 76, 78, 80, 81, 82, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 95, 96, 97, 99, 100, 101, 102]
conflicting records on the same key:


,excel_row,Site ID,Date,Temperature (°C),pH,Dissolved Oxygen (mg/L)
27,30,AV-2,2023-10-31 00:00:00,14.6,7.41,8.2
95,98,AV-2,2023-10-31 00:00:00,14.6,7.41,NaN
58,61,AV-3,2023-12-07 00:00:00,20,7.52,7.9
61,64,AV-3,2023-12-07 00:00:00,21,7.79,7.2
66,69,AV-3,2023-12-29 00:00:00,14.1,7.48,6.9
80,83,AV-3,2023-12-29 00:00:00,200.7,7.26,7.2
91,94,AV-3,2023-12-29 00:00:00,15.3,7.9,12.9


--- fish ---
exact duplicate rows: 16
duplicate on ['Site ID', 'Date', 'Species']: 18
exact copies: 16 rows repeat an earlier row -> [47, 75, 76, 77, 78, 80, 81, 83, 84, 85, 86, 87, 88, 89, 91, 92]
conflicting records on the same key:


,excel_row,Site ID,Date,Species,Count,Avg. Size (cm)
41,44,AV-1,2023-10-08 00:00:00,Shortfin Eel,3,38.8
76,79,AV-1,2023-10-08 00:00:00,Shortfin Eel,3000,38.8
52,55,AV-2,2023-12-14 00:00:00,Longfin Eel,19,44.9
87,90,AV-2,2023-12-14 00:00:00,Longfin Eel,1,4.9


### Interpretation

- **Verbatim copies.**
  - 25 water-quality rows and 16 fish rows repeat an earlier row in every column.
  - All but one in each table sit below row 73 (rows 74–102 and 75–92). The other is row 47.
  - Copies identical to every decimal look like a pasted block, not repeat sampling.
- **Different records under one key.**
  - Water quality, AV-2 31 Oct: rows 30 and 98 (98 lacks DO).
  - Water quality, AV-3 7 Dec: two different readings (rows 61/74 and 64).
  - Water quality, AV-3 29 Dec: **three** different readings (rows 69/82, 83 and 94).
  - Fish: AV-1 8 Oct shortfin eel (rows 44 and 79), and AV-2 14 Dec longfin eel (rows 55 and 90).
- **Not a conflict.** AV-3 on 7 Dec records both īnanga and brown trout. Two species on one visit is a
  legitimate record, which is why the fish key includes the species.
- **Next check.** Most of these conflicts involve a row below 73. The next cell checks whether the other
  records down there are also copies, with a date or a value changed.

In [8]:
import datetime as dt

def show(v):
    return v.strftime('%d %b') if isinstance(v, dt.datetime) else v

def altered_copies(t, cols, last_row=73, min_shared=3):
    """Records below `last_row` that are not verbatim copies, each beside any record in rows 3-`last_row`
    sharing at least `min_shared` of the data columns. Changed cells show 'source -> copy'."""
    exact = t.duplicated(cols, keep='first')
    upper, lower = t[(t.excel_row <= last_row) & ~exact], t[(t.excel_row > last_row) & ~exact]
    out = []
    for _, r in lower.iterrows():
        shared = sum((upper[c].astype(str) == str(r[c])) for c in cols)
        matches = upper[shared >= min_shared]
        if matches.empty:
            out.append({'row': r.excel_row, 'source row': '-', 'shared': '-', **{c: show(r[c]) for c in cols}})
        for i, s in matches.iterrows():
            out.append({'row': r.excel_row, 'source row': s.excel_row, 'shared': f'{shared[i]}/{len(cols)}',
                        **{c: show(s[c]) if str(s[c]) == str(r[c]) else f'{show(s[c])} -> {show(r[c])}'
                           for c in cols}})
    return pd.DataFrame(out)

for name, t, cols in [('water quality', wq, DATA_WQ), ('fish', fish, DATA_FISH)]:
    print(f'--- {name}: records below row 73 that are not verbatim copies ---')
    display(altered_copies(t, cols))

--- water quality: records below row 73 that are not verbatim copies ---


,row,source row,shared,Site ID,Date,Temperature (°C),pH,Dissolved Oxygen (mg/L)
0,77,64,4/5,AV-3,07 Dec -> 12 Dec,21,7.79,7.2
1,79,66,3/5,AV-2,15 Dec -> 01 Dec,18.6,7.67,7.9 -> 79.9
2,83,70,3/5,AV-3,19 Dec -> 29 Dec,20.7 -> 200.7,7.26,7.2
3,94,-,-,AV-3,29 Dec,15.3,7.90,12.9
4,98,30,4/5,AV-2,31 Oct,14.6,7.41,8.2 -> nan


--- fish: records below row 73 that are not verbatim copies ---


,row,source row,shared,Site ID,Date,Species,Count,Avg. Size (cm)
0,74,-,-,AV-3,11/31/2023,Inanga,250,9.6
1,79,44,4/5,AV-1,08 Oct,Shortfin Eel,3 -> 3000,38.8
2,82,22,3/5,AV-2,30 Oct -> 03 Oct,Brown Trout,9 -> 19,18.4
3,90,55,3/5,AV-2,14 Dec,Longfin Eel,19 -> 1,44.9 -> 4.9


### Interpretation

- **The block below row 73 adds nothing new.** Apart from the verbatim copies, every record there but two
  repeats a record above it with one or two changes:
  - **a date moved:** water quality 77 (7 → 12 Dec), 79 (15 → 1 Dec) and 83 (19 → 29 Dec); fish 82
    (30 Oct → 03 Oct, digits swapped);
  - **digits inserted or dropped:** DO 7.9 → 79.9, 20.7 → 200.7 °C, count 3 → 3000, count 9 → 19, and
    19 fish of 44.9 cm → 1 fish of 4.9 cm;
  - **a value blanked:** water quality 98 is row 30 without its DO.
- **Two records have no source:** water quality 94 (DO 12.9 mg/L) and fish 74 (`11/31/2023`, 250 īnanga).
- **Why it matters.** Three altered copies would otherwise pass as real surveys: AV-3 on 12 Dec, AV-2 on
  1 Dec, and a brown-trout survey at AV-2 on 3 Oct, which would be the only date before 6 Oct and the
  only fish record without a water reading.
- **The clean data is therefore rows 3–73 without the copy at row 47:** 70 rows, each pairing a water
  reading with the fish record from the same visit.

## Data-integrity errors and same-day readings

The copies and the invalid date are data-integrity errors. The two same-day pairs are not errors; each
is resolved by a stated rule.

| Excel row(s) | Problem | Decision | Evidence |
|---|---|---|---|
| 25 WQ and 16 fish rows | Verbatim copies | **remove the copy, keep the first** | Identical to every decimal: a pasted block, not new samples |
| WQ 77, 79, 83, 98; fish 79, 82, 90 | Altered copies | **remove the copy, keep its source row** | Each repeats a row above it with a date moved, digits changed or a value blanked (table above). Kept, they would add three surveys that never happened (AV-3 12 Dec, AV-2 1 Dec, AV-2 3 Oct) |
| Fish 74 (AV-3) | Invalid date `11/31/2023` | **remove the record** | No source row. Its 250 īnanga at 9.6 cm also fall outside every other īnanga record (counts 3–30, sizes 6.9–8.5 cm). Without a valid date it can't be placed in a month or matched to a visit |
| WQ 61 and 64 (AV-3, 7 Dec) | Two readings on one day | **average into one visit, keeping `n_readings`** | Both sit in the paired block, each beside its own fish record, so they may be two samples taken that day |
| WQ 69 and 94 (AV-3, 29 Dec) | Two readings on one day | **keep row 69; remove row 94** (next section) | Row 69 is in the paired block (and was copied at 82); row 94 has no source |

- **Judgement call: rows 64 and 77.** They are one reading under two dates, 7 and 12 Dec. Every other
  altered copy moved its date, so row 77 is treated as the copy. If row 64's date is the error instead,
  AV-3 has one reading on 7 Dec (row 61) and one on 12 Dec (with the brown trout), and the same-day pair
  disappears. Either way the reading counts once, not twice.
- **Alternatives tested** in the sensitivity analysis (`03`):
  - re-dating row 64 to 12 Dec adds an AV-3 visit (70 in all), raises December's mean temperature by
    0.1 °C and the maximum to 21.0 °C;
  - taking the first 7 Dec reading instead of the mean.

  Neither changes a test result.

In [9]:
from src.data.quality import outlier_report

NUM_WQ = ['Temperature (°C)', 'pH', 'Dissolved Oxygen (mg/L)']
num = wq[NUM_WQ].apply(pd.to_numeric, errors='coerce')
display(outlier_report(num))

# outlier_report counts flags; the decision needs the rows. Physical limits first, then IQR on the raw
# values, then z-scores recomputed AFTER removing impossible values (one huge error inflates the sd and
# hides the others -- the masking effect).
PLAUSIBLE = {'Temperature (°C)': (0, 30), 'pH': (4, 10), 'Dissolved Oxygen (mg/L)': (0, 20)}
for col, (lo, hi) in PLAUSIBLE.items():
    s = num[col]
    impossible = (s < lo) | (s > hi)
    q1, q3 = s.quantile([0.25, 0.75])
    iqr_flag = (s < q1 - 1.5 * (q3 - q1)) | (s > q3 + 1.5 * (q3 - q1))
    clean = s.mask(impossible)
    z_flag = ((clean - clean.mean()) / clean.std()).abs() > 3
    hit = impossible | iqr_flag | z_flag
    if hit.any():
        print(col)
        display(wq.loc[hit, ['excel_row', 'Site ID', 'Date', col]]
                  .assign(impossible=impossible[hit], iqr=iqr_flag[hit], z_after_removal=z_flag[hit]))
    else:
        print(f'{col}: nothing flagged')

print('fish counts and sizes by species (copies included):')
fish_num = fish.assign(**{c: pd.to_numeric(fish[c], errors='coerce') for c in ['Count', 'Avg. Size (cm)']})
display(fish_num.groupby('Species', dropna=False)[['Count', 'Avg. Size (cm)']].agg(['min', 'median', 'max', 'count']))

Values flagged. Flagged is not the same as wrong -- decide and record why.


,column,n,iqr_flagged,z_flagged,lower_fence,upper_fence,min,max
0,Temperature (°C),100,1,1,9.5375,23.4375,13.20,200.7
1,pH,99,0,0,6.8375,8.0975,7.04,7.9
2,Dissolved Oxygen (mg/L),99,2,1,5.3000,10.5000,6.50,79.9


Temperature (°C)


,excel_row,Site ID,Date,Temperature (°C),impossible,iqr,z_after_removal
80,83,AV-3,2023-12-29 00:00:00,200.7,True,True,False


pH: nothing flagged
Dissolved Oxygen (mg/L)


,excel_row,Site ID,Date,Dissolved Oxygen (mg/L),impossible,iqr,z_after_removal
76,79,AV-2,2023-12-01 00:00:00,79.9,True,True,False
91,94,AV-3,2023-12-29 00:00:00,12.9,False,True,True


fish counts and sizes by species (copies included):


Count                    Avg. Size (cm)                   
               min median   max count            min median   max count
Species                                                                
Brown Trout      3   11.0    22    23           16.6   18.1  19.5    23
Inanga           3   20.5   250    30            6.9    7.6   9.6    30
Longfin Eel      1    6.0    19    21            4.9   42.7  44.9    20
Shortfin Eel     3    6.0  3000    14           38.8   40.2  42.1    14
NaN             26   26.0    26     2            8.1    8.1   8.1     2

## Critical anomalies and statistical outliers

| Excel row | Value | Classification | Decision | Evidence |
|---|---|---|---|---|
| WQ 83 (AV-3) | **200.7 °C** | Critical anomaly | **remove** (an altered copy of row 70, 20.7 °C) | Water boils at 100 °C |
| WQ 79 (AV-2) | DO **79.9 mg/L** | Critical anomaly | **remove** (an altered copy of row 66, 7.9 mg/L) | Even pure oxygen saturates water at about 45 mg/L at 18.6 °C; water in equilibrium with air holds 9.35 mg/L |
| Fish 79 (AV-1) | count **3000** | Data-integrity error | **remove** (an altered copy of row 44, count 3) | Extreme on its own; the source row holds the true value |
| Fish 90 (AV-2) | 1 longfin eel of **4.9 cm** | Data-integrity error | **remove** (an altered copy of row 55, 19 fish of 44.9 cm) | Extreme on its own; the source row holds the true value |
| WQ 94 (AV-3, 29 Dec) | DO **12.9 mg/L** (129% saturation) | Statistical outlier | **remove** | No source row. Flagged by IQR and, once 79.9 is removed, by z-score. It is the only reading above 98% saturation, and it conflicts with row 69 on the same day |

- **Judgement call: row 94.** The value is not impossible. Midday photosynthesis can push DO above
  saturation, and its high pH (7.90) fits that. If it is genuine, AV-3 swung by about 6 mg/L within one
  day. Without sampling times, a high and a low reading can't be told apart from an error; continuous
  loggers would settle it.
- **Alternative tested** in the sensitivity analysis (`03`): keeping row 94, averaged with row 69, lifts
  that visit from 6.9 to 9.9 mg/L, so 11 visits rather than 12 fall below 7 mg/L, and 20 rather than 21
  below 75% saturation. No test result changes.

In [10]:
from src.data.quality import quality_report
from src.visualization.presentation import save_table

_ = save_table(quality_report(wq[DATA_WQ].apply(lambda c: pd.to_numeric(c, errors='coerce') if c.name in NUM_WQ else c),
                              subset=KEY_WQ), '01_quality_water')
_ = save_table(quality_report(fish_num[DATA_FISH], subset=KEY_FISH), '01_quality_fish')

PROFILE
rows: 100   columns: 5
memory: 13.1 KB

dtypes:
float64    3
str        1
object     1

MISSING
complete rows: 98 of 100 (98.0%)
                         missing  missing_pct
pH                             1          1.0
Dissolved Oxygen (mg/L)        1          1.0

DUPLICATES
exact duplicate rows: 25
duplicate on ['Site ID', 'Date']: 29

OUTLIERS (flagged, not removed)
Values flagged. Flagged is not the same as wrong -- decide and record why.
                 column   n  iqr_flagged  z_flagged  lower_fence  upper_fence   min   max
       Temperature (°C) 100            1          1       9.5375      23.4375 13.20 200.7
                     pH  99            0          0       6.8375       8.0975  7.04   7.9
Dissolved Oxygen (mg/L)  99            2          1       5.3000      10.5000  6.50  79.9

SUMMARY (one row per column)
                 column   dtype  non_null  missing  missing_pct  unique             example  unique_ratio note  iqr_flagged  z_flagged
                Si

## What this data can and cannot answer

**Can answer** (October to December 2023, three sites):
- How temperature, pH and DO differ between sites and change over the three months.
- How far DO sits below saturation, and how often it falls, indicatively, into the lower bands of New
  Zealand's National Policy Statement for Freshwater Management (NPS-FM; Ministry for the Environment, 2020).
- Which species were recorded where and when, and whether counts track water quality. There are about
  70 visits, but only 11–25 records per species.

**Cannot answer:**
- **Causation.** The visits are observational; season, flow, time of day and survey effort are uncontrolled.
- **Absence.** No visit records zero fish, so presence/absence can't be modelled.
- **Effort-adjusted abundance.** Survey effort isn't recorded, so a count of 20 may not mean twice a count of 10.
- **Daily minima.** There are no sampling times or continuous data, and NPS-FM bands are defined on
  daily minima.
- **Long-term trends.** The organisation describes several years of monitoring, but this file covers three months.
- **Site order** along the river.

**Next:** `02` applies these decisions, derives oxygen saturation and the taxon flags, summarises every
problem in the data validation and management matrix, and builds the figures. `03` tests the hypotheses
stated in advance.

## References

Ministry for the Environment. (2020). *National Policy Statement for Freshwater Management 2020*. New Zealand
Government. https://gazette.govt.nz/notice/id/2020-go3443